# 1) Notebook Outline

1. **Setup + reproducibility + artifact directory checks**
2. **Auto-detect MS COCO directories and annotation file**
3. **Load COCO captions, build image↔caption tables, and persist artifacts**
4. **Tokenization pipeline with start/end tokens + saved tokenizer config**
5. **Build tf.data caption training samples with padding and split**
6. **ResNet101 encoder feature extractor sanity check (7x7x2048 → 49x2048)**
7. **Bahdanau attention + CNN encoder projection + LSTM decoder classes**
8. **Training step, checkpoint manager, and resume-safe training loop**
9. **Inference: attention-based greedy decoding for one image**
10. **Optional FLAN-T5-small caption refinement**
11. **End-to-end demo and qualitative outputs**


## 2) Setup and global config


In [ ]:
# This cell installs/imports essentials, sets seeds, and prepares artifact paths.  # SAFE TO RERUN
import os
import json
import random
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ARTIFACT_DIR = Path('/kaggle/working/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Artifacts directory: {ARTIFACT_DIR.resolve()}')
print('TensorFlow version:', tf.__version__)


## 3) Detect COCO dataset paths (no hardcoded assumptions)


In [ ]:
# This cell auto-detects COCO paths from Kaggle's attached datasets under /kaggle/input and prints selected paths.  # SAFE TO RERUN
from pathlib import Path

def detect_coco_paths_kaggle():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        raise FileNotFoundError('Expected /kaggle/input to exist in Kaggle runtime.')

    ann_candidates = sorted(input_root.rglob('captions_train2017.json'))

    # Prefer real train2017 folders that contain image files
    img_candidates = []
    for p in sorted(input_root.rglob('train2017')):
        if p.is_dir() and (any(p.glob('*.jpg')) or any(p.glob('*.jpeg'))):
            img_candidates.append(p)

    if not ann_candidates:
        raise FileNotFoundError('Could not find captions_train2017.json under /kaggle/input')
    if not img_candidates:
        raise FileNotFoundError('Could not find a train2017 image directory with .jpg/.jpeg files under /kaggle/input')

    annotation_path = ann_candidates[0]
    image_dir = img_candidates[0]
    return annotation_path, image_dir

# Show the discovered dataset files (compact preview)
all_input_files = sorted(str(x) for x in Path('/kaggle/input').rglob('*') if x.is_file())
print('Total files discovered under /kaggle/input:', len(all_input_files))
for f in all_input_files[:20]:
    print(' -', f)
if len(all_input_files) > 20:
    print(' ... (truncated)')

ANNOTATION_PATH, IMAGE_DIR = detect_coco_paths_kaggle()
print('Selected annotation file:', ANNOTATION_PATH)
print('Selected image directory:', IMAGE_DIR)


## 4) Load annotations, build mappings, and save artifacts


In [ ]:
# This cell builds caption mappings and persists them as pickle/json to avoid recomputation.  # RUN ONCE
import json
import pickle
from collections import defaultdict

mapping_file = ARTIFACT_DIR / 'caption_mapping.pkl'
meta_file = ARTIFACT_DIR / 'caption_meta.json'

if mapping_file.exists() and meta_file.exists():
    print('Loading cached caption mapping artifacts...')
    with open(mapping_file, 'rb') as f:
        data = pickle.load(f)
    image_to_captions = data['image_to_captions']
    all_image_files = data['all_image_files']
    image_ids_to_file = data['image_ids_to_file']
else:
    print('Building caption mappings from raw COCO JSON...')
    with open(ANNOTATION_PATH, 'r') as f:
        coco = json.load(f)

    image_ids_to_file = {im['id']: im['file_name'] for im in coco['images']}
    image_to_captions = defaultdict(list)
    for ann in coco['annotations']:
        file_name = image_ids_to_file.get(ann['image_id'])
        if file_name is not None:
            image_to_captions[file_name].append(ann['caption'].strip())

    all_image_files = sorted(image_to_captions.keys())
    with open(mapping_file, 'wb') as f:
        pickle.dump({'image_to_captions': dict(image_to_captions), 'all_image_files': all_image_files, 'image_ids_to_file': image_ids_to_file}, f)
    with open(meta_file, 'w') as f:
        json.dump({'num_images_with_captions': len(all_image_files), 'num_total_captions': int(sum(len(v) for v in image_to_captions.values()))}, f, indent=2)

image_files_on_disk = {p.name for p in IMAGE_DIR.glob('*.jpg')}
caption_keys = set(all_image_files)
overlap = image_files_on_disk.intersection(caption_keys)
print('Images on disk:', len(image_files_on_disk))
print('Images in caption map:', len(caption_keys))
print('Overlap count:', len(overlap))
print('Overlap ratio wrt captions:', round(len(overlap) / max(len(caption_keys), 1), 4))


## 5) Tokenizer + sequence preparation (resume-safe)


In [ ]:
# This cell creates tokenizer artifacts and tokenized captions with <start>/<end> tokens.  # RUN ONCE
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer

TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.pkl'
TOKEN_META_PATH = ARTIFACT_DIR / 'token_meta.json'
TOKENIZED_CAPTION_PATH = ARTIFACT_DIR / 'tokenized_captions.pkl'

TOP_K_VOCAB = 12000
MAX_LEN = 52

if TOKENIZER_PATH.exists() and TOKEN_META_PATH.exists() and TOKENIZED_CAPTION_PATH.exists():
    print('Loading tokenizer/tokenized caption artifacts...')
    with open(TOKENIZER_PATH, 'rb') as f:
        tokenizer = pickle.load(f)
    with open(TOKENIZED_CAPTION_PATH, 'rb') as f:
        token_data = pickle.load(f)
    image_caption_tokens = token_data['image_caption_tokens']
    max_len = token_data['max_len']
else:
    print('Building tokenizer from captions...')
    all_caps = []
    image_caption_tokens = {}
    for fn, caps in image_to_captions.items():
        wrapped = [f'<start> {c.lower()} <end>' for c in caps]
        image_caption_tokens[fn] = wrapped
        all_caps.extend(wrapped)

    tokenizer = Tokenizer(num_words=TOP_K_VOCAB, oov_token='<unk>')
    tokenizer.fit_on_texts(all_caps)
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'

    max_len = min(MAX_LEN, max(len(c.split()) for c in all_caps))
    with open(TOKENIZER_PATH, 'wb') as f:
        pickle.dump(tokenizer, f)
    with open(TOKEN_META_PATH, 'w') as f:
        json.dump({'top_k_vocab': TOP_K_VOCAB, 'max_len': max_len, 'vocab_size_effective': len(tokenizer.word_index)+1}, f, indent=2)
    with open(TOKENIZED_CAPTION_PATH, 'wb') as f:
        pickle.dump({'image_caption_tokens': image_caption_tokens, 'max_len': max_len}, f)

sample_k = next(iter(image_caption_tokens))
sample_caption = image_caption_tokens[sample_k][0]
sample_seq = tokenizer.texts_to_sequences([sample_caption])[0]
print('Sample image key:', sample_k)
print('Sample wrapped caption:', sample_caption)
print('Sample token IDs (truncated):', sample_seq[:15], '...')
print('max_len:', max_len, '| vocab size:', len(tokenizer.word_index)+1)


## 6) Build tf.data pipeline (memory-safe mini subset configurable)


In [ ]:
# This cell creates a train/valid dataset of image paths and caption token sequences.  # RUN ONCE
from sklearn.model_selection import train_test_split
import numpy as np

DATASET_CACHE = ARTIFACT_DIR / 'dataset_splits.pkl'
MAX_IMAGES = 35000
BATCH_SIZE = 64
BUFFER_SIZE = 2000

if DATASET_CACHE.exists():
    print('Loading cached dataset splits...')
    with open(DATASET_CACHE, 'rb') as f:
        ds_cache = pickle.load(f)
    train_items = ds_cache['train_items']
    val_items = ds_cache['val_items']
else:
    print('Building dataset split artifacts...')
    all_items = []
    selected_keys = sorted(list(image_caption_tokens.keys()))[:MAX_IMAGES]
    for fn in selected_keys:
        img_path = str(IMAGE_DIR / fn)
        for cap in image_caption_tokens[fn]:
            seq = tokenizer.texts_to_sequences([cap])[0][:max_len]
            all_items.append((img_path, seq))
    train_items, val_items = train_test_split(all_items, test_size=0.05, random_state=SEED)
    with open(DATASET_CACHE, 'wb') as f:
        pickle.dump({'train_items': train_items, 'val_items': val_items}, f)

print('Train pairs:', len(train_items), '| Valid pairs:', len(val_items))

def pad_seq(seq):
    arr = np.zeros((max_len,), dtype=np.int32)
    arr[:len(seq)] = seq
    return arr

def build_tf_dataset(items, batch_size=BATCH_SIZE, training=True):
    img_paths = [x[0] for x in items]
    seqs = np.stack([pad_seq(x[1]) for x in items])
    ds = tf.data.Dataset.from_tensor_slices((img_paths, seqs))

    def _load_image(path, cap):
        im = tf.io.read_file(path)
        im = tf.image.decode_jpeg(im, channels=3)
        im = tf.image.resize(im, (224, 224))
        im = tf.keras.applications.resnet.preprocess_input(im)
        return im, cap

    if training:
        ds = ds.shuffle(BUFFER_SIZE, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = build_tf_dataset(train_items, training=True)
val_ds = build_tf_dataset(val_items, training=False)

for imgs, caps in train_ds.take(1):
    print('Training batch image shape:', imgs.shape)
    print('Training batch caption shape:', caps.shape)


## 7) ResNet101 encoder feature shape sanity check


In [ ]:
# This cell verifies encoder outputs (7,7,2048) and reshaped attention features (49,2048).  # SAFE TO RERUN
base_model = tf.keras.applications.ResNet101(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
cnn_backbone = tf.keras.Model(base_model.input, base_model.output)

for batch_imgs, _ in train_ds.take(1):
    feat_map = cnn_backbone(batch_imgs[:2], training=False)
    feat_seq = tf.reshape(feat_map, (tf.shape(feat_map)[0], -1, tf.shape(feat_map)[-1]))
    print('Encoder feature map shape:', feat_map.shape)
    print('Encoder sequence shape:', feat_seq.shape)


## 8) Define Bahdanau attention + decoder architecture


In [ ]:
# This cell defines model classes for attention-based captioning.  # SAFE TO RERUN
EMBED_DIM = 256
UNITS = 512
VOCAB_SIZE = len(tokenizer.word_index) + 1

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = self.V(tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = tf.reduce_sum(attention_weights * features, axis=1)
        return context_vector, attention_weights

class CNNEncoder(tf.keras.Model):
    def __init__(self, embed_dim):
        super().__init__()
        self.fc = tf.keras.layers.Dense(embed_dim)

    def call(self, x):
        return tf.nn.relu(self.fc(x))

class RNNDecoder(tf.keras.Model):
    def __init__(self, embed_dim, units, vocab_size):
        super().__init__()
        self.units = units
        self.attention = BahdanauAttention(units)
        self.embedding = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc1 = tf.keras.layers.Dense(units)
        self.fc2 = tf.keras.layers.Dense(vocab_size)

    def call(self, x, features, hidden):
        context_vector, attention_weights = self.attention(features, hidden)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x)
        x = self.fc1(output)
        x = tf.reshape(x, (-1, x.shape[2]))
        return self.fc2(x), state_h, state_c, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

encoder = CNNEncoder(EMBED_DIM)
decoder = RNNDecoder(EMBED_DIM, UNITS, VOCAB_SIZE)
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
print('Model objects initialized.')


## 9) Training loop with checkpointing (resume-safe)


In [ ]:
# This cell trains the model and resumes from latest checkpoint if present.  # RUN ONCE
CKPT_DIR = ARTIFACT_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

checkpoint = tf.train.Checkpoint(encoder=encoder, decoder=decoder, optimizer=optimizer, cnn_backbone=cnn_backbone)
ckpt_manager = tf.train.CheckpointManager(checkpoint, str(CKPT_DIR), max_to_keep=5)

if ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint)
    print('Restored from checkpoint:', ckpt_manager.latest_checkpoint)
else:
    print('No checkpoint found. Starting training from scratch.')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    return tf.reduce_mean(loss_ * mask)

@tf.function
def train_step(img_tensor, target):
    total_loss = 0.0
    batch_size = tf.shape(target)[0]
    hidden = decoder.reset_state(batch_size=batch_size)
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * tf.cast(batch_size, tf.int32), 1)

    with tf.GradientTape() as tape:
        features = cnn_backbone(img_tensor, training=False)
        features = tf.reshape(features, (tf.shape(features)[0], -1, tf.shape(features)[-1]))
        features = encoder(features)
        for i in tf.range(1, tf.shape(target)[1]):
            predictions, hidden, _, _ = decoder(dec_input, features, hidden)
            total_loss += loss_function(target[:, i], predictions)
            dec_input = tf.expand_dims(target[:, i], 1)

    vars_ = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(total_loss, vars_)
    optimizer.apply_gradients(zip(grads, vars_))
    return total_loss / tf.cast(tf.shape(target)[1], tf.float32)

EPOCHS = 3
STEPS_PER_EPOCH = 350

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for step, (img_batch, cap_batch) in enumerate(train_ds.take(STEPS_PER_EPOCH)):
        batch_loss = train_step(img_batch, cap_batch)
        epoch_loss += float(batch_loss.numpy())
        if step % 50 == 0:
            print(f'Epoch {epoch+1} Step {step}/{STEPS_PER_EPOCH} Loss {batch_loss.numpy():.4f}')

    ckpt_path = ckpt_manager.save()
    print(f'Epoch {epoch+1} done. Mean loss: {epoch_loss / STEPS_PER_EPOCH:.4f}. Saved: {ckpt_path}')


## 10) Inference function (base caption generation)


In [ ]:
# This cell defines greedy decoding inference for one image.  # SAFE TO RERUN
idx_to_word = tokenizer.index_word
word_to_idx = tokenizer.word_index

def load_image_for_infer(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    return tf.keras.applications.resnet.preprocess_input(img)

def generate_caption(image_path, max_length=None):
    max_length = max_length or max_len
    img = tf.expand_dims(load_image_for_infer(image_path), 0)

    features = cnn_backbone(img, training=False)
    features = tf.reshape(features, (1, -1, tf.shape(features)[-1]))
    features = encoder(features)

    hidden = decoder.reset_state(batch_size=1)
    dec_input = tf.expand_dims([word_to_idx['<start>']], 0)

    out_words = []
    for _ in range(max_length):
        preds, hidden, _, _ = decoder(dec_input, features, hidden)
        pred_id = int(tf.argmax(preds[0]).numpy())
        pred_word = idx_to_word.get(pred_id, '<unk>')
        if pred_word == '<end>':
            break
        out_words.append(pred_word)
        dec_input = tf.expand_dims([pred_id], 0)
    return ' '.join(out_words)

print('Inference function ready.')


## 11) Optional FLAN-T5-small refinement


In [ ]:
# This cell adds optional FLAN-T5-small refinement with lazy loading.  # SAFE TO RERUN
FLAN_READY = False
flan_tokenizer, flan_model = None, None

def load_flan_t5_small():
    global FLAN_READY, flan_tokenizer, flan_model
    if FLAN_READY:
        return
    from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM
    model_name = 'google/flan-t5-small'
    flan_tokenizer = AutoTokenizer.from_pretrained(model_name)
    flan_model = TFAutoModelForSeq2SeqLM.from_pretrained(model_name)
    FLAN_READY = True

def refine_caption_with_flan(base_caption):
    if not FLAN_READY:
        load_flan_t5_small()
    prompt = f'Rewrite this image caption to be more natural and concise: {base_caption}'
    inputs = flan_tokenizer(prompt, return_tensors='tf', truncation=True)
    outputs = flan_model.generate(**inputs, max_new_tokens=40)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)

print('FLAN utilities ready; model loads on first refinement call.')


## 12) End-to-end demo


In [ ]:
# This cell runs base captioning and displays an image with generated caption.  # SAFE TO RERUN
import matplotlib.pyplot as plt

sample_img_path, _ = val_items[0]
base_caption = generate_caption(sample_img_path)
print('Sample image:', sample_img_path)
print('Base caption:', base_caption)

# Optional refinement
# refined_caption = refine_caption_with_flan(base_caption)
# print('Refined caption:', refined_caption)

img = plt.imread(sample_img_path)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title(base_caption)
plt.show()
